In [ ]:
from PIL import Image
import torch
import torch.nn as nn
from torchvision.models import *
import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
resnetik = resnet18(pretrained=True)
resnetik.fc = nn.Identity()
resnetik.eval()
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def read_img(path):
    img = Image.open(path).convert("RGB").resize((224, 224))
    x = np.array(img, dtype=np.float32).transpose((2, 0, 1)) / 255.0
    x = torch.tensor(x[None])
    x = (x - mean) / std
    return x
def norm(img):
    img = Image.fromarray(img).convert("RGB").resize((224, 224))
    x = np.array(img, dtype=np.float32).transpose((2, 0, 1)) / 255.0
    x = torch.tensor(x[None])
    x = (x - mean) / std
    return x
X1 = []
y = []
for i in tqdm.tqdm(range(201)):
    X = read_img('train_data (4)/img_'+str(i)+'.png')
    X1.append(resnetik(X).detach().numpy())
    y.append(1)
for i in tqdm.tqdm(range(201)):
    id1 = i
    id2 = np.random.randint(0, 201)
    while id2 == id1:
        id2 = np.random.randint(0, 201)
    Xfirst = np.array(Image.open('train_data (4)/img_'+str(id1)+'.png'), dtype=np.float32)
    Xsecond = np.array(Image.open('train_data (4)/img_'+str(id2)+'.png'), dtype=np.float32)
    X = np.array(np.round((Xfirst + Xsecond)/2), dtype=np.uint8)
    # print(X.shape)
    X1.append(resnetik(norm(X)).detach().numpy())
    y.append(0)
    # plt.imshow(Xfirst.transpose(1, 2, 0))
    # plt.show()
    # plt.imshow(X.transpose(1, 2, 0))
    # plt.show()
    # break
X1 = np.array(X1)

# print(X1.shape)
# print(X2.shape)

c:\Users\dauza\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\dauza\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 201/201 [00:18<00:00, 10.75it/s]


In [116]:
X2 = []
for i in tqdm.tqdm(range(2000)):
    X = read_img('test_data (2)/img_'+str(i)+'.png')
    X2.append(resnetik(X).detach().numpy())
X2 = np.array(X2)

100%|██████████| 2000/2000 [02:56<00:00, 11.32it/s]


In [117]:
X1.shape

(402, 1, 2048)

In [118]:
from sklearn.linear_model import *
model = LogisticRegression()
model.fit(X1.reshape((X1.shape[0], -1)), y)
y2 = model.predict(X2.reshape((X2.shape[0], -1)))

In [119]:
sub = pd.DataFrame({'subtaskID' : [1] * len(X2), 'datapointID' : range(len(X2)), 'answer' : y2})
sub.to_csv('submission.csv', index=False)